# 07 — Transformers From Scratch: Attention, Masks, Decoder-Only LMs, and Generation

Goal: implement decoder-only Transformers and understand chatbot-style generation.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Decoder-only LM (educational)

We implement:
- token embeddings
- sinusoidal positional encoding
- TransformerEncoder blocks with a causal mask
- LM head producing vocab logits

This is an instructional scaffold; production LLMs add: dropout, RMSNorm, rotary embeddings, KV-cache, etc.

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=256, max_len=256, pad_id=0):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.tok_emb(input_ids)
        x = self.pos(x)
        T = input_ids.size(1)
        causal = torch.triu(torch.ones(T, T, device=input_ids.device), diagonal=1).bool()
        x = self.tr(x, mask=causal)
        return self.lm_head(x)

## 2. Simple tokenizer + dataset construction (char-level)

Use your own corpora for better results.

In [ ]:

class SimpleCharTok:
    def __init__(self, texts):
        chars = set()
        for t in texts: chars.update(list(t))
        self.itos = ["<pad>","<bos>","<eos>"] + sorted(chars)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
    def encode(self, text, add_bos=False, add_eos=False):
        ids=[]
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(c, self.pad_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids
    def decode(self, ids):
        out=[]
        for i in ids:
            if 0 <= i < len(self.itos) and self.itos[i] not in ("<pad>","<bos>","<eos>"):
                out.append(self.itos[i])
        return "".join(out)

corpus = [
    "Hello world!",
    "Hello PyTorch. PyTorch makes tensors and models.",
    "Tokenization turns text into tokens, then ids."
]
tok = SimpleCharTok(corpus)

def make_lm_data(texts, tokenizer, block_size=64):
    ids=[]
    for t in texts:
        ids += tokenizer.encode(t, add_bos=True, add_eos=True)
    ids = torch.tensor(ids, dtype=torch.long)
    xs, ys = [], []
    step = block_size
    for i in range(0, len(ids) - block_size - 1, step):
        xs.append(ids[i:i+block_size])
        ys.append(ids[i+1:i+block_size+1])
    return torch.stack(xs), torch.stack(ys)

x, y = make_lm_data(corpus * 200, tok, block_size=64)
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x,y), batch_size=32, shuffle=True)

lm = TinyDecoderLM(vocab_size=len(tok.itos), pad_id=tok.pad_id, max_len=128).to(device)
opt = torch.optim.AdamW(lm.parameters(), lr=3e-4)

for epoch in range(3):
    lm.train()
    total = 0.0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        logits = lm(xb)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()
    print("epoch", epoch, "loss", total/len(dl))

## 3. Generation: temperature + top-k sampling

In [ ]:

@torch.inference_mode()
def sample_next(logits, temperature=1.0, top_k=None):
    logits = logits / max(temperature, 1e-8)
    if top_k is not None:
        v, ix = torch.topk(logits, k=top_k)
        mask = torch.full_like(logits, float("-inf"))
        mask.scatter_(0, ix, v)
        logits = mask
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=120, temperature=1.0, top_k=50):
    model.eval()
    ids = tokenizer.encode(prompt, add_bos=True)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :].squeeze(0)
        next_id = sample_next(logits, temperature=temperature, top_k=top_k)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == tokenizer.eos_id:
            break
    return tokenizer.decode(x.squeeze(0).tolist())

print(generate(lm, tok, "Hello", temperature=0.9, top_k=20))

## 4. KV-cache concept (important)

In production LLM inference, you do not recompute attention over the entire context each step.
Instead:
- cache past keys/values per layer
- compute new keys/values for the new token
- attend to cached + new

This reduces generation from O(T^2) to O(T) per token for the attention component.
Implementation depends on the attention module you use.